In [3]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

documents = [
    Document(
        page_content="""
        환불 규정

        상품 구매 후 7일 이내에는 환불을 요청할 수 있습니다.
        단, 상품을 사용하거나 훼손한 경우 환불이 제한될 수 있습니다.

        배송 규정

        상품은 결제 완료 후 영업일 기준 2~3일 이내 배송됩니다.
        """
    )
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=30,
    chunk_overlap=5,
)

chunks = splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    print(i, chunk.page_content)

0 환불 규정
1 상품 구매 후 7일 이내에는 환불을
2 환불을 요청할 수 있습니다.
3 단, 상품을 사용하거나 훼손한 경우
4 경우 환불이 제한될 수 있습니다.
5 배송 규정
6 상품은 결제 완료 후 영업일 기준
7 기준 2~3일 이내 배송됩니다.


In [8]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

embedding = OpenAIEmbeddings(model="text-embedding-3-small")
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()
print(f'documents len: {len(documents)}')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)
print(f'chunks len: {len(chunks)}')

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding
)
print("FAISS 생성완료")

query = "펀드가 무엇인가요?"
result = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(result):
    print(f"결과: {i}")
    print(f"페이지: {doc.metadata.get('page')}")
    print(doc.page_content)
    print('-' * 80)

documents len: 191
chunks len: 332
FAISS 생성완료
결과: 0
페이지: 28
으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.
--------------------------------------------------------------------------------
결과: 1
페이지: 31
것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점

In [16]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()
# print(documents)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]{question}",
        )
    ]
)

parser = StrOutputParser()

# question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"
question = "대한민국의 수도는 어디인가요?"

retriever_docs = retriever.invoke(question)
# print(retriever_docs)

context = "\n\n".join([doc.page_content for doc in retriever_docs])

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

sdf [Document(id='350f1aea-a770-4207-816f-e231ffba264c', metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2018-12-27T10:04:51+09:00', 'moddate': '2018-12-28T11:19:21+09:00', 'title': '', 'trapped': '/False', 'source': './data/doc/금융투자협회_투자길라잡이_2018.pdf', 'total_pages': 191, 'page': 67, 'page_label': '68'}, page_content='므로 과다매매가 인정 됨\nA종목 매도 관련, ’12. 1. 10 정규장 마감시각 직전의 거래소의 매수호가 잔량은 약 32만주에 달하여 직원이 주문을 \n지정가주문이 아닌 시장가주문으로 하였더라면 전량 매도가 가능한 상황이었음. 지정가주문으로 고객의 ’당일 전부 \n처분‘지시를 따르지 못한 것은 전문가로서 선관주의의무에 충실했다고 보기는 무리가 있음\n다만, B종목 매도와 관려하여서는 문자메세지 및 전화통화로 사후보고를 하였으나 이에 고객의 이의제기 등이 없었던 \n점, 매도자금으로 타종목 매수를 승인한 사실 등을 고려할 때, 증권사 직원의 임의매도로 보기는 어렵고, 설령 임의 \n매도라 하더라도 고객이 묵시적추인을 하였다고 판단 함\n판·단·내·용'), Document(id='f9793d0b-8963-4cc6-87f7-d50e9e809a2d', metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Adobe InDesign CS4 (6.0)', 'creationdate': '2018-12-27T10:04:51+09:00', 'moddate': '2018-12-28T11:19:21+09:00', 'title': '', 'trapped': '/Fals

In [22]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)
chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

questions = [
    "펀드란 무엇인가요?",
    "대한민국의 수도는 어디인가요?",
    "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"
]

for question in questions:
    result = vectorstore.similarity_search_with_score(question, k=3)

    print(f"질문: {question}")

    for i, (doc, score) in enumerate(result):
        print("-" * 100)
        print("검색 결과")
        print("distance: ", score)
        print("page: ", doc.metadata.get("page"))
        print("내용: ", doc.page_content[:500])

질문: 펀드란 무엇인가요?
----------------------------------------------------------------------------------------------------
검색 결과
distance:  1.0027242
page:  28
내용:  으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.
----------------------------------------------------------------------------------------------------
검색 결과
distance:  1.0495431
page:  31
내용:  것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환

In [28]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

question = "펀드의 종류와 특징은 무엇인가요?"

similarity_retriever = vectorstore.as_retriever(
    search_type="similarity", 
    search_kwargs={"k": 3}
)
# print(similarity_retriever)

similarity_docs = similarity_retriever.invoke(question)
# print(similarity_docs) # document형태

mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 3, # 서로 너무 비슷하지 않도록 고려해서 최종적으로 3개를 골라
        "fetch_k": 10, # MMR이 최종 3개를 고르기 전에 먼저 살펴볼 후보의 수
        "lambda_mult": 0.5, # 관련성과 다양성은 50:50으로 봐 (MMR의 진짜 핵심 옵션)
                            # 1.0	관련성 중심 / 0.5	관련성 + 다양성 균형 / 0.0	다양성 중심
    }
)
mmr_docs = mmr_retriever.invoke(question)

print("\n")
print("=" * 80)
print("Similarity Search 결과")
print("=" * 80)

for i, doc in enumerate(similarity_docs):
    print(f"\n----- 결과 {i + 1} -----")
    print("Page: ", doc.metadata.get("page"))
    print(doc.page_content[:500])


# ============================================================
# MMR 결과
# ============================================================
print("\n")
print("=" * 80)
print("MMR Search 결과")
print("=" * 80)

for i, doc in enumerate(mmr_docs):
    print(f"\n----- 결과 {i + 1} -----")
    print("Page: ", doc.metadata.get("page"))
    print(doc.page_content[:500])



Similarity Search 결과

----- 결과 1 -----
Page:  28
으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.

----- 결과 2 -----
Page:  30
사후관리 서비스를 제공합니다. 펀드판매사가 제공하는 보고서에는 분기별 자산운용보고서, 투자자별 펀드잔고 
통보서, 자산보관·관리보고서 등이 있습니다. 
•자산운용보고서 확인 : 자산운용사가 작성하여 판매회사가 송부하는 분기별 자산운용보고서를 활용하여 내가
가입한 펀드가 어떻게 운용되고 있는지를 확인합니다. 자산운용보고서를 통해 운용성과 개요 및 운용 기간 중 
손익, 펀드자산내역, 매매금액 및 매매 회전율, 향후 운용계획 등의 정보가 제공됩니다. 
•‘투자자별 펀드잔고 통보’서비스 활용 : 판매사는 ‘투자자별 펀드잔고 통보’ 서비스를 제공하는데, 이 서비스는
투자자의 펀드잔고, 개별 수익률 정보 등을 전자우편 등을 통해 제공하는 서비스로 모든 투자자에게 제공됩니다. 
다만, 제공을 원하지 않을 경우 별도의 신청을 통하여 제공 받지 않을 수 있습니다.
▒ 적립식/거치식 펀

In [32]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

load_dotenv()

EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)
vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

question = "펀드란 무엇인가?"

all_results = vectorstore.similarity_search(
    question,
    k=3
)
# print(all_results)

for i, doc in enumerate(all_results):
    print(f"\n----- 결과 {i + 1} -----")
    print("[Page] ", doc.metadata.get("page"))
    print("[Content]")
    print(doc.page_content[:500])

filtered_results = vectorstore.similarity_search(
    question,
    k=3,
    filter={"page": 50} # metadata의 page가 50인 것만 검색 대상으로 해라
)

print()

if not filtered_results:
    print("조건에 맞는 문서를 찾지 못했습니다.")
else:
    for i, doc in enumerate(filtered_results):
        print(f"\n----- 결과 {i + 1} -----")
        print("[Page]", doc.metadata.get("page"))
        print("[Content]")
        print(doc.page_content[:500])


----- 결과 1 -----
[Page]  28
[Content]
으로 지급받도록 되어 있으므로, 펀드의 수익률에 큰 영향을 미치게 됩니다. 따라서 펀드 선택시 보수와 
수수료가 어떻게 되는지도 살펴보는 것이 좋습니다.
1) 펀드 투자요령
Q.
A.
펀드 가입시 고려해야 할 사항에는 어떤 것들이 있습니까?
펀드는 원금손실가능성이 있는 실적배당상품으로 가입시 고려하여야 할 기본적인 사항들을 예를 들어
보면 다음과 같습니다.
① 투자기간, 투자유형, 투자성향 등 투자목적에 맞는 펀드를 선택해야 합니다.
② 펀드가입시기 결정 등과 관련하여 주식시장 및 채권시장의 흐름을 면밀히 살펴야 합니다.
③  운용회사나 펀드매니저에 따라 운용성과에 차이가 많이 발생할 수 있으므로 운용회사의 과거 실적 
및 운용전략, 펀드매니저의 운용성향 등도 사전에 확인하여야 합니다.
④ 갑작스런 자금수요에 대비하여 사전에 중도환매가 가능한지를 꼭 확인하여야 합니다.
⑤ 판매담당인력의 전문성 및 판매상품의 다양성 등을 고려하여 판매회사를 골라야 합니다.

----- 결과 2 -----
[Page]  31
[Content]
것보다는 장기적인 안목을 가지고 결정하는 것이 바람직합니다. 판매회사에서 보내주는 각종 보고서의 정보를 
참고하여 내 펀드의 현재 상태를 점검하고 금융시장의 현황을 수시로 관찰하면서 향후 경제 전망 정보 등을 
참고하여 적절한 환매시점을 결정합니다. 
•펀드 환매절차 : 환매를 신청하고자 할 경우 환매시 적용되는 기준가격, 환매대금지급일, 환매수수료(환매제한
기간중에는 투자자가 환매수수료를 부담)에 관한 정보를 미리 확인하여야 합니다. 이 정보들은 투자설명서를 
통해 확인 가능합니다. 투자설명서는 펀드 가입시 제공되며, 금융투자협회의 인터넷 전자공시서비스 페이지
(dis.kofia.or.kr)에서도 확인할 수 있습니다. 환매는 직접 지점방문, 전화 또는 인터넷 사이트를 통해 신청할 수 
있으며, 환매대금을 지정한 계좌로 입금 또는 창구에서 현금으로 직접 지급받을 수 있습니

In [33]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

model = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embeddings)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
bm25_retriever = BM25Retriever.from_documents(chunks, k=3)

hybrid_retriever = EnsembleRetriever(
    retrievers = [vector_retriever, bm25_retriever],
    weight = [0.5, 0.5]
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)
context = "\n\n".join(doc.page_content for doc in retrieved_docs)
parser = StrOutputParser()

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print(result)

모바일폰을 사용한 금융거래 시 유의해야 할 점은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기.
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않기.
3. 금융거래 비밀번호를 안전하게 관리하기.
4. 모바일폰 분실·도난 시 모바일폰 금융서비스 사용 중지하기.
5. 모바일폰 교체·수리 전 중요 정보 삭제하기.
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기.
7. 모바일폰 사용환경을 임의로 변경하지 않기.
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기.
9. 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기.
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용 시 주의하기.


In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

model = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vectorstore = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",   # 다국어 지원, 한국어 OK
    model_kwargs={"device": "mps"},         # Mac GPU 사용 (Windows 강의의 "cuda" 자리)
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

question = "모바일폰을 사용한 금융거래 시 어떤 점을 유의해야 하는가?"

retrieved_docs = hybrid_retriever.invoke(question)
print("검색된 후보 문서 : ", len(retrieved_docs))

reranked_docs = reranker.compress_documents(documents=retrieved_docs, query=question)

final_docs = reranked_docs   # top_n=3에서 이미 잘려서 나옴

for i, doc in enumerate(final_docs, start=1):
    print(f"\n--- 최종 문서 {i} ---")
    print("[Page]", doc.metadata.get("page"))
    print(doc.page_content[:500])

context = "\n\n".join([doc.page_content for doc in final_docs])

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        (
            "user",
            "[질문]\n{question}",
        ),
    ]
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 13771.58it/s]


검색된 후보 문서 :  16

--- 최종 문서 1 ---
[Page] 25
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
 금융투자상품에 투자하기   25
① 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기 
② 모바일폰이나 인터넷에 금융정보를 저장하지 않기
③ 금융거래 비밀번호를 안전하게 관리하기
④ 모바일폰 분실·도난시 모바일폰 금융서비스 사용 중지하기.
⑤ 모바일폰 교체·수리 전 중요 정보 삭제하기
⑥ 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기
⑦ 모바일폰 사용환경을 임의로 변경하지 않기
⑧ 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기
⑨ 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기
⑩ 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용시 주의하기
모바일폰 금융거래 10계명
모바일폰은 언제, 어디에서나 사용하는 필수품이 되었습니다. 이제는 금융거래도 모바일폰을 이용하여 언제, 
어디서든지 할 수 있습니다.  
여러 금융기관에서는 금리

--- 최종 문서 2 ---
[Page] 16
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
   분쟁조정사례·판례집16
ಿীై੗ೞӝǍh੟h੉ಿীై੗ೞӝǍh੟h੉
나날이 복잡해져 가는 금융투자상품에 대해 정확
하게 이해하기 위하여 투자설명서를 꼼꼼히 체크
하여야 하며, 이해가 가지 않는 부분이 있다면 금
융투자회사 직원에게 설명을 요청하는 것이 좋습
니다.
특히, 계좌개설 시 반드시 투자설명서 및 약관을 
교부받아 적어도 투자기간 동안 보관해 두어야 합
니다.
금융투자상품 가입 시 작성하는 서류에 투자자 본
인이 자필로 서명하거나 기재하는 문구가 여러가
지 있습니다. 특히 투자설명서의 이해 여부를 확인하기 위한 본인기재사항은 투자자가 해당 금융투자상품을 
이해하고 투자한다는 것을 선언하는 것으로 사후 법적다툼이 발생할 때 판단하는 1차적인 자료로 활용될 수 
있으므로 그 내용을 충분히 확인한 후 작성하

In [5]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

question = "모바일 폰을 사용한 금융 거래 시 어떤 점을 유의해야 하는가?"

prompt = ChatPromptTemplate.from_template("""
    다음 질문에 대해 문서 검색에 사용할 수 있는 서로 다른 검색 질문 3래를 만들어주세요.

    원래 질문
    {question}

    조건:
    - 원래 질문의 의미는 유지할 것
    - 서로 다른 표현을 사용할 것
    - 검색에 도움이 되는 핵심 단어를 포함할 것
    - 번호나 설명없이 검색 질문만 한 줄씩 출력할 것
""")

parser = StrOutputParser()

cross_encoder = HuggingFaceCrossEncoder(
    model_name="BAAI/bge-reranker-v2-m3",   # 다국어 지원, 한국어 OK
    model_kwargs={"device": "mps"},         # Mac GPU 사용 (Windows 강의의 "cuda" 자리)
)
reranker = CrossEncoderReranker(model=cross_encoder, top_n=3)

chain = prompt | llm | parser

generated_queries_text = chain.invoke({"question": question})
generated_queries = [
    query.strip() for query in generated_queries_text.split("\n") if query.strip()
]

all_docs = []

for i, query in enumerate(generated_queries, start=1):
    print(query)
    docs = hybrid_retriever.invoke(query)

    print("검색된 문서 수:", len(docs))
    all_docs.extend(docs) # 검색 결과를 하나의 리스트에 모읍니다.

print("\n전체 검색 결과:", len(all_docs))

unique_docs = {}

for doc in all_docs:
    page = doc.metadata.get("page")

    if page not in unique_docs:
        unique_docs[page] = doc

unique_docs = list(unique_docs.values())
print("중복 제거 후 문서 수 : ", len(unique_docs))

reranked_docs = reranker.compress_documents(documents=unique_docs, query=question)

final_docs = reranked_docs

context = "\n\n".join(doc.page_content for doc in final_docs)

prompt = ChatPromptTemplate(
    [
        (
            "system",
            """
            당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.

            반드시 아래 제공된 문서 내용만 근거로 답변하세요.

            문서에 답이 없다면
            "제공된 문서에서 답을 찾을 수 없습니다."
            라고 답변하세요.

            [문서 내용]
            {context}
            """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)

chain = prompt | llm | parser

result = chain.invoke({"context": context, "question": question})

print("result: ", result)


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 12675.41it/s]


모바일 기기로 금융 거래를 할 때 주의해야 할 점은 무엇인가요?
검색된 문서 수: 20
금융 거래를 위해 모바일 폰을 사용할 때 유의해야 할 사항은 어떤 게 있나요?
검색된 문서 수: 20
모바일 폰을 통한 금융 거래 시 필수적으로 고려해야 할 요소는 무엇인가요?
검색된 문서 수: 19

전체 검색 결과: 59
중복 제거 후 문서 수 :  37
result:  모바일 폰을 사용한 금융 거래 시 유의해야 할 사항은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스를 이용하기.
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않기.
3. 금융거래 비밀번호를 안전하게 관리하기.
4. 모바일폰 분실·도난 시 모바일폰 금융서비스 사용 중지하기.
5. 모바일폰 교체·수리 전 중요 정보 삭제하기.
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기.
7. 모바일폰 사용환경을 임의로 변경하지 않기.
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기.
9. 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’를 수시로 변경하기.
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용 시 주의하기.


In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL, temperature=0)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)

cross_encoder = HuggingFaceCrossEncoder(
    model_name=RERANK_MODEL,
    model_kwargs={"device": "mps"},
)
reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=10,   # ← 아래 설명 참고
)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

rewrite_prompt = ChatPromptTemplate.from_template(
    """
다음 사용자의 질문을 문서 검색에 적합한 형태로
다시 작성하세요.

조건:
- 질문의 원래 의미를 유지하세요.
- 불필요한 표현은 제거하세요.
- 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
- 질문 하나만 출력하세요.
- 설명이나 번호는 출력하지 마세요.

사용자 질문:
{question}
"""
)

rewrite_chain = rewrite_prompt | llm | StrOutputParser()


def evaluate_hit_rate(
    question,
    relevant_pages,
    k=5,
    use_rerank=True # true/false로 결과를 찍어서 비교해보면 리랭크가 필요한 지 알 수 있음
):
    rewritten_question = rewrite_chain.invoke({"question": question})

    retrieved_docs = hybrid_retriever.invoke(rewritten_question)

    if use_rerank:
        retrieved_docs = reranker.compress_documents(
            documents=retrieved_docs,
            query=question,
        )

    top_docs = retrieved_docs[:k]

    retrieved_pages = [doc.metadata.get("page") for doc in top_docs]

    hit = any(page in relevant_pages for page in retrieved_pages)

    return hit, retrieved_pages, rewritten_question


evaluation_data = [
    {
        "question": "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?",
        "relevant_pages": [25, 16, 21],
    },
]
hits = []

for item in evaluation_data:
    hit, pages, rewritten_question = evaluate_hit_rate(
        question=item["question"],
        relevant_pages=item["relevant_pages"],
        k=5,
    )

    hits.append(int(hit))

    print("=" * 60)

    print("원래 질문:")
    print(item["question"])

    print("\nRewrite된 질문:")
    print(rewritten_question)

    print("\n정답 페이지:")
    print(item["relevant_pages"])

    print("\n검색된 Top 5 페이지:")
    print(pages)

    print("\nHit@5:")
    print(int(hit))

hit_rate = sum(hits) / len(hits)

print("\n" + "=" * 60)
print(f"최종 Hit Rate@5: {hit_rate:.4f}")
print("=" * 60)

In [9]:
import os
import re

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
RERANK_MODEL = "BAAI/bge-reranker-v2-m3"
PDF_PATH = "./data/doc/금융투자협회_투자길라잡이_2018.pdf"

llm = ChatOpenAI(model=MODEL, temperature=0)
embedding = OpenAIEmbeddings(model=EMBEDDING_MODEL)
cross_encoder = HuggingFaceCrossEncoder(
    model_name=RERANK_MODEL,
    model_kwargs={"device": "mps"},
)
reranker = CrossEncoderReranker(
    model=cross_encoder,
    top_n=10,
)

loader = PyPDFLoader(PDF_PATH)

documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
chunks = splitter.split_documents(documents)

vector_store = FAISS.from_documents(documents=chunks, embedding=embedding)

vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
bm25_retriever = BM25Retriever.from_documents(chunks, k=10)
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5]
)

parser = StrOutputParser()

rewrite_prompt = ChatPromptTemplate.from_template(
    """
    다음 사용자의 질문을 문서 검색에 적합한 형태로
    다시 작성하세요.
    
    조건:
    - 질문의 원래 의미를 유지하세요.
    - 불필요한 표현은 제거하세요.
    - 문서 검색에 도움이 되는 핵심 키워드를 포함하세요.
    - 질문 하나만 출력하세요.
    - 설명이나 번호는 출력하지 마세요.
    
    사용자 질문:
    {question}
    """
)
rewrite_chain = rewrite_prompt | llm | parser

answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
        "system",
        """
        당신은 금융투자 안내 문서를 기반으로 질문에 답변하는 도우미입니다.
        
        반드시 아래 제공된 문서 내용만 근거로 답변하세요.
        
        문서에 답이 없다면
        "제공된 문서에서 답을 찾을 수 없습니다."
        라고 답변하세요.
        
        [문서 내용]
        {context}
        """,
        ),
        ("user", "[질문]\n{question}"),
    ]
)
answer_chain = answer_prompt | llm | parser

context_eval_prompt = ChatPromptTemplate.from_template(
    """
    당신은 RAG 검색 품질 평가자입니다.
    
    사용자 질문과 검색된 문서를 보고,
    검색된 문서가 질문에 얼마나 관련 있는지 평가하세요.
    
    점수:
    0.0 = 전혀 관련 없음
    0.5 = 일부 관련
    1.0 = 매우 관련 있음
    
    반드시 숫자 하나만 출력하세요.
    
    [질문]
    {question}
    
    [Context]
    {context}
    """
)
context_eval_chain = context_eval_prompt | llm | parser

faithfulness_prompt = ChatPromptTemplate.from_template(
    """
    당신은 RAG 답변의 사실성을 평가하는 평가자입니다.
    
    답변에 포함된 내용이 제공된 Context에 의해
    뒷받침되는지를 평가하세요.
    
    Context에 없는 내용을 답변이 추가했다면
    점수를 낮게 주세요.
    
    점수:
    0.0 = 대부분 근거 없음
    0.5 = 일부만 근거 있음
    1.0 = 모든 내용이 Context에 근거함
    
    반드시 숫자 하나만 출력하세요.
    
    [Context]
    {context}
    
    [Answer]
    {answer}
    """
)
faithfulness_chain = faithfulness_prompt | llm | parser

answer_relevance_prompt = ChatPromptTemplate.from_template(
    """
    당신은 RAG 답변의 질문 관련성을 평가하는 평가자입니다.
    
    사용자의 질문과 답변을 비교하여
    답변이 질문에 얼마나 적절하게 답했는지 평가하세요.
    
    점수:
    0.0 = 질문에 답하지 못함
    0.5 = 일부만 답함
    1.0 = 질문에 정확하게 답함
    
    반드시 숫자 하나만 출력하세요.
    
    [질문]
    {question}
    
    [Answer]
    {answer}
    """
)
answer_relevance_chain = answer_relevance_prompt | llm | parser

question = "모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?"

rewritten_question = rewrite_chain.invoke({"question": question})

print("=" * 60)
print("원래 질문:")
print(question)
print("\nRewrite 질문:")
print(rewritten_question)

retrieved_docs = hybrid_retriever.invoke(rewritten_question)

reranked_docs = reranker.compress_documents(
    documents=retrieved_docs,
    query=question
)

context = "\n\n".join(doc.page_content for doc in reranked_docs)

answer = answer_chain.invoke(
    {
        "context": context,
        "question": question,
    }
)

print("\n" + "=" * 60)
print("최종 Answer:")
print(answer)

context_score = context_eval_chain.invoke(
    {
        "question": question,
        "context": context,
    }
)

faithfulness_score = faithfulness_chain.invoke(
    {
        "context": context,
        "answer": answer
    }
)

answer_relevance_score = answer_relevance_chain.invoke(
    {
        "question": question,
        "answer": answer
    }
)

def parse_score(text: str) -> float | None:
    """LLM 출력에서 0~1 사이 숫자 하나를 꺼낸다. 실패하면 None"""
    match = re.search(r"\d+(\.\d+)?", text)
    if not match:
        return None
    score = float(match.group())
    return score if 0.0 <= score <= 1.0 else None

print("\n" + "=" * 60)
print("RAG Evaluation")
print("-" * 60)

print("Context Relevance:", parse_score(context_score))
print("Faithfulness:", parse_score(faithfulness_score))
print("Answer Relevance:", parse_score(answer_relevance_score))

print("=" * 60)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 11003.16it/s]


원래 질문:
모바일 금융거래를 할 때 주의해야 할 사항은 무엇인가?

Rewrite 질문:
모바일 금융거래 주의사항은 무엇인가?

최종 Answer:
모바일 금융거래를 할 때 주의해야 할 사항은 다음과 같습니다:

1. 금융회사가 안내하는 공식 배포처를 확인하여 금융서비스 이용하기.
2. 모바일폰이나 인터넷에 금융정보를 저장하지 않기.
3. 금융거래 비밀번호를 안전하게 관리하기.
4. 모바일폰 분실·도난 시 모바일폰 금융서비스 사용 중지하기.
5. 모바일폰 교체·수리 전 중요 정보 삭제하기.
6. 휴대폰 문자서비스(SMS), 일회용비밀번호(OTP) 이용하기.
7. 모바일폰 사용환경을 임의로 변경하지 않기.
8. 모바일폰 보안업데이트를 정기적으로 수행하고 바이러스 검사하기.
9. 모바일폰 ‘잠금기능’을 설정하고 ‘잠금비밀번호’는 수시로 변경하기.
10. 출처가 불분명하거나 보안설정 없는 무선랜(Wi-Fi) 사용 시 주의하기.

RAG Evaluation
------------------------------------------------------------
Context Relevance: 1.0
Faithfulness: 1.0
Answer Relevance: 1.0


In [ ]:
_